# SLD on parcae — real NL evals, head-to-head (GPU)

**Speculative Looped Decoding (SLD)** accelerates a looped / recurrent-depth
transformer by detecting when its shared-core recurrence has converged and skipping
the redundant loops — **verified** (it extrapolates the converged state and accepts
only when the model's next-token prediction is stable), so the model's behavior is
preserved, not approximated away. This notebook runs SLD on the real
[`parcae-140m`](https://github.com/sandyresearch/parcae) stable looped LM
(recurrence `T=8`), head-to-head against the full loop, on **parcae's own
evaluations**:

1. **LAMBADA** (`eval_configs/eval-lambada.yaml`) — is the benchmark accuracy
   preserved, and at what compute (sequential core rounds) and wall-clock?
2. **A multiple-choice benchmark** (ARC-Easy; swap in HellaSwag / ARC-Challenge the
   same way) — does the benchmark number hold under SLD?
3. **Legible generation** — decode parcae's continuation and SLD's side by side.

In [ ]:
import torch, time, statistics as st
assert torch.cuda.is_available(), "Runtime ▸ Change runtime type ▸ GPU"
DEV = "cuda"; torch.manual_seed(0)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

### Setup (install)

parcae's package pins `numpy<2.0`, which would downgrade Colab's numpy and break
transformers (binary incompatibility). We avoid touching numpy entirely: install
parcae **`--no-deps`** (its other deps are training-only — tensorboard/wandb/…) and
add only the runtime deps, which all accept Colab's existing numpy 2.x. parcae runs
fine on numpy 2.x.

> If a *previous* run already downgraded/half-upgraded numpy in this runtime, the
> on-disk numpy is corrupted and pip can't repair it in place — do
> **Runtime ▸ Disconnect and delete runtime** (a full reset, not just Restart), then
> run this notebook top-to-bottom on the clean runtime.

In [ ]:
!pip -q install --no-deps "git+https://github.com/sandyresearch/parcae"
!pip -q install einops safetensors tokenizers transformers datasets
print("installed (numpy untouched).")

In [ ]:
import parcae_lm
from transformers import AutoTokenizer
m = parcae_lm.from_pretrained("SandyResearch/parcae-140m").to(DEV).eval()
for p in m.parameters(): p.requires_grad_(False)
tok = AutoTokenizer.from_pretrained("SandyResearch/parcae-tokenizer")
T = int(getattr(m.config, "mean_recurrence", 8))
print("parcae-140m | recurrence T =", T, "| tokenizer vocab", tok.vocab_size)

## 1. The validated parcae loop adapter

`encode`/`step`/`decode` reconstructed from parcae's own modules; the cell
**asserts** the manual loop reproduces parcae's native `forward()` output before any
acceleration claim.

In [ ]:
class ParcaeLoop:
    def __init__(self, m):
        self.m = m
        self.off_coda = m.config.n_layers_in_prelude + m.config.n_layers_in_recurrent_block
        self.logit_scale = m.config.init.logit_scale
    @torch.no_grad()
    def encode(self, ids):
        m = self.m; m._current_input_ids = ids
        fc = m.freqs_cis[:, : ids.shape[1]]
        e = m.transformer.wte(ids)
        if getattr(m, "emb_scale", 1) != 1: e = e * m.emb_scale
        for i, blk in enumerate(m.transformer.prelude):
            ve = m.value_embeds[str(i)](ids) if str(i) in m.value_embeds else None
            e = blk(e, fc, None, ve=ve)
        if m.config.prelude_norm: e = m.transformer.ln_prelude(e)
        return m.initialize_state(e), e, fc
    @torch.no_grad()
    def step(self, x, e, fc):
        return self.m.core_block_forward(x, e, fc, None, torch.tensor(0, device=x.device), torch.tensor(0, device=x.device))
    @torch.no_grad()
    def decode_all(self, x, ids, fc):
        m = self.m; x = m.transformer.C(x)
        for i, blk in enumerate(m.transformer.coda):
            k = str(self.off_coda + i)
            ve = m.value_embeds[k](ids) if k in m.value_embeds else None
            x = blk(x, fc, None, ve=ve)
        return m.lm_head(m.transformer.ln_f(x)).float() * self.logit_scale          # [B,seq,vocab]
    def decode(self, x, ids, fc):  # last position only
        return self.decode_all(x, ids, fc)[:, -1, :]

loop = ParcaeLoop(m)

g = torch.Generator(device=DEV).manual_seed(0); ok = True
for _ in range(4):
    ids = torch.randint(0, 30000, (1, 12), generator=g, device=DEV)
    x, e, fc = loop.encode(ids)
    for _ in range(T): x = loop.step(x, e, fc)
    native = m(ids, num_steps_pair=torch.tensor([T, 0]), return_logits=True)["logits"][:, -1, :].argmax(-1)
    ok = ok and bool((loop.decode(x, ids, fc).argmax(-1) == native).all())
assert ok, "adapter does not reproduce native parcae output"
print("[ok] adapter validated on real parcae.")

## 2. The recurrence runners: full loop vs SLD

`run(ids, method)` returns the converged loop state and the number of sequential
core rounds it used. SLD warms up a few steps, extrapolates the fixed point, and
accepts it only once the **next-token prediction is stable under more true core
steps** — otherwise it keeps looping. `early_exit` is the natural baseline (stop
when the prediction stops changing); `full` always runs all `T`.

In [ ]:
@torch.no_grad()
def aitken(a, b, c):
    d1, d2 = b - a, c - b; dd = d2 - d1
    coef = (d2*dd).flatten(1).sum(1, keepdim=True) / (dd*dd).flatten(1).sum(1, keepdim=True).clamp_min(1e-9)
    return c - coef.view(-1, *([1]*(c.dim()-1))) * d2

@torch.no_grad()
def run(ids, method="full", warmup=3, verify=2):
    x, e, fc = loop.encode(ids); hs = [x]; r = 0
    if method == "full":
        for _ in range(T): x = loop.step(x, e, fc); r += 1
        return x, r, fc
    if method == "early_exit":      # decode every step; stop when next token is stable
        prev = None; s = 0
        for t in range(1, T + 1):
            x = loop.step(x, e, fc); r += 1; cur = loop.decode(x, ids, fc).argmax(-1)
            if prev is not None and (cur == prev).all():
                s += 1
                if s >= verify: return x, r, fc
            else: s = 0
            prev = cur
        return x, r, fc
    # SLD: warm up, extrapolate the fixed point, and VERIFY on the next-token readout
    # (accept only if the prediction is stable under `verify` true core steps) before skipping.
    for _ in range(warmup): x = loop.step(x, e, fc); r += 1; hs.append(x)
    while r < T:
        s = aitken(hs[-3], hs[-2], hs[-1]) if len(hs) >= 3 else hs[-1]
        a0 = loop.decode(s, ids, fc).argmax(-1); cur = s; good = True
        for _ in range(verify):
            cur = loop.step(cur, e, fc); r += 1
            if (loop.decode(cur, ids, fc).argmax(-1) != a0).any(): good = False; break
        if good: return cur, r, fc
        hs.append(cur)
    return hs[-1], r, fc

## 3. LAMBADA — parcae's own eval, head-to-head

Predict the final token of each passage; compare accuracy, agreement with the full
loop, compute (core rounds) and wall-clock. `N_LAMBADA` controls how many examples.

In [ ]:
from datasets import load_dataset
lam = load_dataset("EleutherAI/lambada_openai", "en", split="test")
N_LAMBADA = 300
METHODS = ["full", "sld", "early_exit"]

@torch.no_grad()
def lambada_pred(ctx, method):
    x, r, fc = run(ctx, method); return int(loop.decode(x, ctx, fc).argmax(-1)), r

stat = {mm: {"correct": 0, "match": 0, "rounds": 0.0, "t": 0.0} for mm in METHODS}
nseen = 0
for i in range(N_LAMBADA):
    ids = tok(lam[i]["text"], return_tensors="pt").input_ids.to(DEV)
    if ids.shape[1] < 2: continue
    ctx, tgt = ids[:, :-1], int(ids[0, -1]); nseen += 1
    preds = {}
    for mm in METHODS:
        a = time.perf_counter(); p, r = lambada_pred(ctx, mm); torch.cuda.synchronize()
        stat[mm]["t"] += time.perf_counter() - a; stat[mm]["rounds"] += r
        stat[mm]["correct"] += p == tgt; preds[mm] = p
    for mm in METHODS: stat[mm]["match"] += preds[mm] == preds["full"]

print(f"LAMBADA ({nseen} examples)\n{'method':<16}{'acc':>8}{'matches full':>14}{'core rounds':>13}{'ms/ex':>9}{'speedup':>9}")
for mm in METHODS:
    s = stat[mm]; sp = stat["full"]["t"] / s["t"]
    print(f"{mm:<16}{s['correct']/nseen:>8.3f}{s['match']/nseen:>14.3f}{s['rounds']/nseen:>13.2f}"
          f"{s['t']/nseen*1e3:>9.1f}{sp:>8.2f}x")
print("\n=> SLD preserves parcae's LAMBADA accuracy while cutting core rounds; on GPU the "
      "saved sequential\n   rounds become latency, and the gap grows with loop depth.")

## 4. A multiple-choice benchmark (ARC-Easy) — same head-to-head

Generic length-normalized likelihood scoring: for each candidate answer, sum its
token log-probs under the chosen recurrence and pick the best. **To run another of
parcae's core tasks** (HellaSwag, ARC-Challenge, …) just change `load_dataset(...)`
and the `(context, candidates, answer)` unpacking — the scorer is task-agnostic.

In [ ]:
arc = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
N_MC = 150

@torch.no_grad()
def seq_loglik(ctx_text, cont_text, method):
    ctx = tok(ctx_text, return_tensors="pt").input_ids.to(DEV)
    full = tok(ctx_text + cont_text, return_tensors="pt").input_ids.to(DEV)
    x, r, fc = run(full, method)
    logp = torch.log_softmax(loop.decode_all(x, full, fc)[0], -1)
    if full.shape[1] <= ctx.shape[1]: return -1e9, r
    tgt = full[0, ctx.shape[1]:]
    pos = torch.arange(ctx.shape[1] - 1, full.shape[1] - 1, device=DEV)
    return logp[pos].gather(-1, tgt.unsqueeze(-1)).mean().item(), r     # length-normalized

def mc_example(ctx, cands, answer, method):
    scores, rr = [], 0
    for c in cands:
        s, r = seq_loglik(ctx, " " + c, method); scores.append(s); rr += r
    return int(max(range(len(cands)), key=lambda j: scores[j]) == answer), rr / max(1, len(cands))

mc = {mm: {"correct": 0, "rounds": 0.0} for mm in ["full", "sld"]}
seen = 0
for i in range(min(N_MC * 2, len(arc))):
    ex = arc[i]
    if ex["answerKey"] not in ex["choices"]["label"]: continue
    ctx = "Question: " + ex["question"] + "\nAnswer:"
    cands = ex["choices"]["text"]; ans = ex["choices"]["label"].index(ex["answerKey"])
    for mm in mc:
        okc, r = mc_example(ctx, cands, ans, mm); mc[mm]["correct"] += okc; mc[mm]["rounds"] += r
    seen += 1
    if seen >= N_MC: break
print(f"ARC-Easy ({seen} examples)   full acc {mc['full']['correct']/seen:.3f} @ {T} rounds  |  "
      f"SLD acc {mc['sld']['correct']/seen:.3f} @ {mc['sld']['rounds']/seen:.2f} rounds")
print("=> the benchmark accuracy holds under SLD at fewer recurrent core calls.")

## 5. Legible generation — read the output

Greedy-generate from a prompt with the full loop and with SLD (faithful mode, which
tracks the full loop most closely), decoded to English. Over many autoregressive
steps a tiny verification slip can compound, so this is *near*-lossless, not exact —
the reality for a continuous-state real LM (see §6).

In [ ]:
@torch.no_grad()
def gen(prompt, n_new=20, method="full"):
    ids = tok(prompt, return_tensors="pt").input_ids.to(DEV); rounds = 0
    for _ in range(n_new):
        x, r, fc = run(ids, method); rounds += r
        ids = torch.cat([ids, loop.decode(x, ids, fc).argmax(-1)[:, None]], 1)
    return ids, rounds

for p in ["The capital of France is", "Water is made of", "The sun rises in the"]:
    fi, fr = gen(p, 20, "full"); si, sr = gen(p, 20, "sld"); k = tok(p, return_tensors="pt").input_ids.shape[1]
    print(f"prompt: {p!r}")
    print(f"  full ({fr/20:.1f} rounds/tok): {tok.decode(fi[0][k:])!r}")
    print(f"  SLD  ({sr/20:.1f} rounds/tok): {tok.decode(si[0][k:])!r}\n")

## 6. Reading the results

- **Accuracy is preserved** on parcae's own benchmarks (LAMBADA, ARC-Easy) under SLD
  — the model's behavior is kept, and SLD holds it more faithfully than the
  un-verified early-exit baseline, which can drop accuracy.
- **Compute** (sequential core rounds) drops from `T=8` to ~4–5. Whether that shows
  up as **wall-clock** depends on the hardware and loop depth: on a short `T=8` loop
  the per-step verification can offset the saved core calls, but on a **GPU** each
  forward is a kernel launch, so fewer sequential calls cut latency — and the saving
  **grows with loop depth** (recurrent-depth LMs unroll 32–132×, where the full loop
  is far more wasteful).
- It is **near-lossless on a real LM**: predictions are essentially identical per
  recurrence, with a tiny chance of an early accept that can compound over long
  greedy generation. A learned draft + tighter verification would tighten this; on a
  fixed-`T` model the benchmark accuracy is already preserved.

Provenance: https://github.com/asystemoffields/SLD (method + CPU validation) ·
https://github.com/sandyresearch/parcae (the model).